**Librerie necessarie per runnare il progetto**

In [ ]:
#librerie di base
import keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import metrics
import seaborn as sns

#per scaricare il dataset
from google.colab import drive
import gdown

#per normalizzare i dati ed effettuare riduzione di dimensionalità
from sklearn.preprocessing import StandardScaler

#librerie necessarie a trainare e validare l'albero decisionale
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

#libreria per il calcolo prestazioni computazionali dei modelli
from time import time

# librerie per metriche di valutazione dei modelli
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

#librerie per trainare e validare la rete neurale
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.utils
from tensorflow.keras.optimizers import SGD



#installazioni dei moduli necessari all'ambiente python
!pip install gdown
!pip install tensorflow
!pip install -U scikit-learn

**Carichiamo il dataset e normalizziamo i dati che lo necessitano**

In [ ]:
# Diamo accesso al nostro google drive che conterrà il dataset che utilizzeremo in questo laboratorio

drive.mount('/content/drive/')
file_path = '/content/drive/MyDrive/creditdataset.csv'

# ID necessario ad identificare univocamente il file da caricare
file_id = "15mtL9uBpOlMrpK68_PF6B_jrP32wAPOo"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "creditdataset.csv", quiet=False)

# Leggi il file
dataset = pd.read_csv("creditdataset.csv")

# Separazione delle caratteristiche (features) e del target
featuresDataFrame = dataset.drop(columns=['default payment next month', 'ID'])
targetsDataFrame = dataset['default payment next month']

# Elenco delle colonne da normalizzare
variabili_da_normalizzare = [
    'LIMIT_BAL',  # Importo del credito
    'AGE',  # Età
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',  # Importi delle fatture
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'         # Importi dei pagamenti
]

# Controllo che le variabili siano presenti nel DataFrame
variabili_da_normalizzare = [col for col in variabili_da_normalizzare if col in featuresDataFrame.columns]
print(featuresDataFrame.head())

# Normalizzazione solo delle variabili specificate
scaler = StandardScaler()
featuresDataFrame[variabili_da_normalizzare] = scaler.fit_transform(featuresDataFrame[variabili_da_normalizzare])

# Controllo del risultato
print(featuresDataFrame.head())


MessageError: Error: credential propagation was unsuccessful

Si notano alcune anomalie nei dati delle istanze.

In particolare:
- Nella colonna PAY_N (che indica i mesi di ritardo nel pagamento) il dataset riporta -1 come pagamento in orario, e inoltre compaiono valori come -2, che non ha un significato spiegato nella documentazione del dataset. Noi portiamo questi valori a 0 per consistenza.
- Nella colonna dello stato coniugale i valori possibili sono 1=married, 2=single, 3=others, ma compaiono altri valori rispetto a quelli indicati. Portiamo tutti i valori differenti al valore 3 (altro)
- Nella colonna education i valori possibili sono 1=graduate school, 2=university, 3=high school, 4=others. Ancora una volta portiamo qualsiasi valore differente a 4 (altro)

In [ ]:
featuresDataFrame.dtypes

In [ ]:
pay_columns = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in pay_columns:
    featuresDataFrame.loc[featuresDataFrame[col].isin([-2, -1, 0]), col] = 0

fil = (featuresDataFrame.EDUCATION == 5) | (featuresDataFrame.EDUCATION == 6) | (featuresDataFrame.EDUCATION == 0)
featuresDataFrame.loc[fil, 'EDUCATION'] = 4
featuresDataFrame.loc[featuresDataFrame.MARRIAGE == 0, 'MARRIAGE'] = 3

In [ ]:
# Suddivisione in istanze di Training e di Test
Xtrain, Xtest, Ytrain, Ytest = train_test_split(featuresDataFrame, targetsDataFrame, test_size=0.3, random_state=1)

In [ ]:
Xtrain

**Uno sguardo alla distribuzione delle etichette**

In [ ]:
# Crea un grafico a barre per visualizzare la distribuzione della variabile target
plt.bar(Ytrain.unique(), Ytrain.value_counts())
plt.xlabel('Default (0 = No, 1 = Si)')  # Etichetta per l'asse x
plt.xticks([0, 1])
plt.ylabel('Conteggio')  # Etichetta per l'asse y
plt.title('Distribuzione')  # Titolo del grafico
plt.show()  # Mostra il grafico

Bisognerà tener conto del fatto che il dataset è sbilanciato

**Iniziamo la fase di training e validazione dell'albero decisionale al fine di finetunare gli iperametri**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(Xtrain, Ytrain, test_size=0.2, random_state=42)

# Definisci la griglia di parametri per il grid search
param_grid = {
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Crea il modello di decision tree
model = DecisionTreeClassifier(random_state=42)

# Crea il grid search con validazione incrociata (5-fold)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)

# Fitta il grid search sui dati di allenamento
grid_search.fit(X_train, y_train)

# Stampa i migliori iperparametri
print(f"Migliori iperparametri: {grid_search.best_params_}")

# Usa il modello con i migliori iperparametri per fare predizioni
best_tree = grid_search.best_estimator_

# Fai le predizioni sul validation set
y_pred = best_tree.predict(X_val)

# Calcola e stampa l'accuratezza sul validation set
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy sul validation set: {accuracy:.3f}")



In [ ]:
# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(best_tree, filled=True, ax=ax)
plt.plot()

In [ ]:
# predizione del nuovo modello
Ypred = best_tree.predict(Xtest)

cm = confusion_matrix(Ytest, Ypred)

print("Confusion matrix of the pruned model:\n", cm)
print("\nAccuracy of the pruned model:", cm.diagonal().sum() / cm.sum())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()



In questo caso otteniamo una accuracy del circa 80%.

Tuttavia, nel nostro dataset il 20% dei clienti è in default, di conseguenza se costruissimo un modello che assegna sempre il valore 0 alla predizione, otterremmo comunque un'accuratezza dell'80%. In questo senso, l'accuratezza può essere una metrica fuorviante per valutare la qualità del nostro modello.

Una metrica più adeguata è l'F1-score, che tiene conto dei falsi positivi, dei falsi negativi, ecc.



In [ ]:
path = best_tree.cost_complexity_pruning_path(Xtrain, Ytrain)
ccp_alphas = path.ccp_alphas

# Addestra l'albero decisionale con diversi valori di complessità
train_accuracy = []
test_accuracy = []
for complexity in ccp_alphas:
    clf = DecisionTreeClassifier(max_depth=5, ccp_alpha=complexity)
    clf.fit(Xtrain, Ytrain)
    train_accuracy.append(clf.score(Xtrain, Ytrain))
    test_accuracy.append(clf.score(Xtest, Ytest))

# Plotta l'accuratezza del modello in funzione del parametro di complessità
plt.plot(ccp_alphas, train_accuracy, label='Training Accuracy')
plt.plot(ccp_alphas, test_accuracy, label='Test Accuracy')
plt.xlabel('Complexity Parameter')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Complexity Parameter')
plt.xscale('log')
plt.legend()
plt.show()

Aggiungiamo il nuovo iperparametro: il plot sugerisce il valore 10E-1.9

In [ ]:
# il nostro alberò avrà massima profondità 5 e utilizziamo la migliore ccp_alpha precedentemente trovata
best_params = best_tree.get_params()
prunedModel = DecisionTreeClassifier(random_state=42, ccp_alpha=1 / 10**1.9, min_samples_leaf = best_params["min_samples_leaf"], min_samples_split = best_params["min_samples_split"], criterion=best_params["criterion"])

start_time = time()
prunedModel = prunedModel.fit(Xtrain, Ytrain)
end_time = time()
discriminatoryTreeTime = end_time - start_time

# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(prunedModel, filled=True, ax=ax)
plt.plot()

Il modello all'apparenza sembrerebbe eccessivamente semplificato, si ottiene lo split di un solo attributo, tuttavia la precisione e l'F1-score sono alti

In [ ]:
YPredTest = prunedModel.predict(Xtest)
cm = confusion_matrix(Ytest, YPredTest)

print("Confusion matrix of the pruned model:\n", cm)

accuracy_train_test = accuracy_score(Ytest, YPredTest)
precision = precision_score(Ytest, YPredTest, pos_label=0)
recall = recall_score(Ytest, YPredTest,pos_label=0)
f1 = f1_score(Ytest, YPredTest, pos_label=0)

# Stampa le prestazioni del modello
print('\nAccuracy:', accuracy_train_test)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()

In [ ]:
print(classification_report(YPredTest, Ytest))

Proviamo allora con un modello più complesso

In [ ]:
# il nostro alberò avrà massima profondità 5 e utilizziamo la migliore ccp_alpha precedentemente trovata
best_params = best_tree.get_params()
lessPrunedModel = DecisionTreeClassifier( random_state=42, ccp_alpha=1 / 10**3.4, min_samples_leaf = best_params["min_samples_leaf"], min_samples_split = best_params["min_samples_split"], criterion=best_params["criterion"])

start_time = time()
lessPrunedModel = lessPrunedModel.fit(Xtrain, Ytrain)
end_time = time()
discriminatoryTreeTime = end_time - start_time

# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(lessPrunedModel, filled=True, ax=ax)
plt.plot()

In [ ]:
YPredTestLess = lessPrunedModel.predict(Xtest)
cm = confusion_matrix(Ytest, YPredTestLess)

print("Confusion matrix of the pruned model:\n", cm)

accuracy_train_test = accuracy_score(Ytest, YPredTestLess)
precision = precision_score(Ytest, YPredTestLess, pos_label=0)
recall = recall_score(Ytest, YPredTestLess,pos_label=0)
f1 = f1_score(Ytest, YPredTestLess, pos_label=0)

# Stampa le prestazioni del modello
print('\nAccuracy:', accuracy_train_test)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print(classification_report(YPredTestLess, Ytest))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()

Il modello diventa piú complesso ma la precisione non migliora significativamente, bisogna indagare sull'attributo splittato dal modello semplificato.

In [ ]:
# Importanza delle feature
importances = best_tree.feature_importances_
# Ordina le feature per importanza decrescente
indices = np.argsort(importances)[::-1]

feature_names = list(featuresDataFrame.columns)

plt.figure(figsize=(10, 6))
plt.title("Importanza delle Feature")
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.xlabel("Feature")
plt.ylabel("Importanza")
plt.tight_layout()
plt.show()

Il grafico conferma quanto emerso in precedenza utilizzando il modello semplificato, una singola feature contiene la stragrande maggioranza di significatività. Per cui uno split è sufficiente ad effettuare una corretta previsione dei labels.

Di seguito il grafico della curva ROC del modello semplificato con ccp_alpha a 10E-1.9, che presenta questa spezzata invece della classica curva poiché la semplificazione effettuata porta il modello ad assegnare uno spazio equiprobabile alle predizioni.

In [ ]:
probabilities = prunedModel.predict_proba(Xtest)[:, 0]
print("Prediction Probabilities: ", probabilities)
print("\n")

# Calcola la curva ROC per i negativi
treeFnr, treeTnr, treeThresholds = roc_curve(Ytest, probabilities, pos_label=0)

tree_roc_auc = roc_auc_score(Ytest, probabilities)

# Disegna la curva ROC
plt.plot(treeFnr, treeTnr, label='tree ROC curve (area = %0.2f)' % tree_roc_auc)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.ylabel('False Negative Rate')
plt.xlabel('True Negative Rate')
plt.title('Receiver operating characteristic (Negative Class)')
plt.legend(loc="lower right")
plt.show()


**Traniamo e validiamo la nostra rete multistrato al fine di definirne l'architettura**

In [ ]:
# Funzione per costruire il modello
def create_model(optimizer='adam', hidden_layer_sizes=(50,)):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))  # Definisci la forma di input
    # Aggiungi layer nascosti
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer per classificazione binaria
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

class KerasModel(BaseEstimator, ClassifierMixin):
    def __init__(self, optimizer='adam', hidden_layer_sizes=(50,)):
        self.optimizer = optimizer
        self.hidden_layer_sizes = hidden_layer_sizes
        self.model = create_model(optimizer=self.optimizer, hidden_layer_sizes=self.hidden_layer_sizes)

    def fit(self, X, y):
        self.model.fit(X, y, epochs=5, batch_size=32, verbose=0)
        return self

    def predict(self, X):
        return (self.model.predict(X) > 0.5).astype("int32")

    def predictProb(self, X):
        return self.model.predict(X)

# Caricamento dei dati (Xtrain e Ytrain devono essere già definiti

keras_model = KerasModel()

# Definisci la griglia di parametri per il grid search
param_grid = {
    'optimizer': ['adam', 'sgd'],
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],  # Configurazioni dei layer nascosti
}

# Esegui il grid search con validazione incrociata (5-fold)
grid_search = GridSearchCV(estimator=keras_model, param_grid=param_grid, cv=3)

# Fitta il grid search sui dati di allenamento
grid_search.fit(X_train, y_train)

# Stampa i migliori iperparametri
print(f"Migliori iperparametri: {grid_search.best_params_}")

# Usa il modello con i migliori iperparametri per fare predizioni
best_net = grid_search.best_estimator_

# Fai le predizioni sul validation set
y_pred = best_net.predict(X_val)

# Calcola e stampa l'accuratezza sul validation set
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy sul validation set: {accuracy:.3f}")



In [ ]:
best_net.model.summary()

In [ ]:
keras.utils.plot_model(best_net.model, show_shapes=True)

Training del modello dummy per ottenere informazioni sul tempo di training richiesto

**Valutiamo la rete multistrato**

In [ ]:
YPredDiscNet = best_net.predict(Xtest)

accuracy_train_test = accuracy_score(Ytest, YPredDiscNet)
precision = precision_score(Ytest, YPredDiscNet, pos_label=0)
recall = recall_score(Ytest, YPredDiscNet, pos_label=0)
f1 = f1_score(Ytest, YPredDiscNet, pos_label=0)

# Stampa le prestazioni del modello
print('Accuracy:', accuracy_train_test)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print(classification_report(YPredDiscNet, Ytest))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()


anche questa volta la precisione pesata è soddisfacente

**Confrontiamo le curve roc dei due modelli concentrandoci sull'etichetta 0 [false]**

In [ ]:
# Perchè vogliamo lavorare sui negativi
YpredNegativa = 1 - best_net.predictProb(Xtest)
print("Prediction Probabilities: ", YpredNegativa)
print("\n")

# Calcola la curva ROC per la classe negativa (pos_label=0)
fnr, tnr, thresholds = roc_curve(Ytest, YpredNegativa, pos_label=0)

# Calcola l'AUC della curva ROC per la classe negativa
roc_auc = roc_auc_score(Ytest, YpredNegativa)


# Disegna la curva ROC per la classe negativa

plt.plot(fnr, tnr, label='MLP ROC curve (area = %0.2f)' % roc_auc)
plt.plot(treeFnr, treeTnr, label='Tree ROC curve (area = %0.2f)' % tree_roc_auc)


plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')  # Linea diagonale
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Negative Rate')
plt.ylabel('True Negative Rate')  # Tasso di veri negativi
plt.title('Receiver Operating Characteristic (Negative Class)')
plt.legend(loc="lower right")
plt.show()


le curve roc sono comparabili per i due modelli in esame.

**sono progettate per problemi complessi e dataset molto grandi, con strutture complicate, nel dataset in esame modelli più semplici possono essere altrettanto efficaci e più efficienti.**

# Ripetiamo il procedimento questa volta senza includere le colonne SEX, EDUCATION e MARRIAGE. In quanto discriminativi.

In [ ]:
featuresDataFrame = featuresDataFrame.drop(columns=['SEX', 'EDUCATION', 'MARRIAGE'])

# Suddivisione in istanze di Training e di Test
Xtrain1, Xtest1, Ytrain1, Ytest1 = train_test_split(featuresDataFrame, targetsDataFrame, test_size=0.3, random_state=1)

In [ ]:
# Suddivisione tra istanze di Training e di Validazione
X_train, X_val, y_train, y_val = train_test_split(Xtrain1, Ytrain1, test_size=0.2, random_state=42)

# Definisci la griglia di parametri per il grid search
param_grid = {
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Crea il modello di decision tree
model = DecisionTreeClassifier(random_state=42)

# Crea il grid search con validazione incrociata (5-fold)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)

# Fitta il grid search sui dati di allenamento
grid_search.fit(X_train, y_train)

# Stampa i migliori iperparametri
print(f"Migliori iperparametri: {grid_search.best_params_}")

# Usa il modello con i migliori iperparametri per fare predizioni
best_friendly_tree = grid_search.best_estimator_

# Fai le predizioni sul validation set
y_pred = best_friendly_tree.predict(X_val)

# Calcola e stampa l'accuratezza sul validation set
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy sul validation set: {accuracy:.3f}")



In [ ]:
# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(best_friendly_tree, filled=True, ax=ax)
plt.plot()

In [ ]:
# predizione del nuovo modello
Ypred12 = best_friendly_tree.predict(Xtest1)

cm = confusion_matrix(Ytest1, Ypred12)

print("Confusion matrix of the pruned model:\n", cm)
print("\nAccuracy of the pruned model:", cm.diagonal().sum() / cm.sum())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()

In [ ]:
path = prunedModel.cost_complexity_pruning_path(Xtrain1, Ytrain1)
ccp_alphas = path.ccp_alphas


# Addestra l'albero decisionale con diversi valori di complessità
train_accuracy = []
test_accuracy = []
for complexity in ccp_alphas:
    clf = DecisionTreeClassifier(max_depth=5, ccp_alpha=complexity)
    clf.fit(Xtrain1, Ytrain1)
    train_accuracy.append(clf.score(Xtrain1, Ytrain1))
    test_accuracy.append(clf.score(Xtest1, Ytest1))

# Plotta l'accuratezza del modello in funzione del parametro di complessità
plt.plot(ccp_alphas, train_accuracy, label='Training Accuracy')
plt.plot(ccp_alphas, test_accuracy, label='Test Accuracy')
plt.xlabel('Complexity Parameter')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Complexity Parameter')
plt.xscale('log')
plt.legend()
plt.show()

In [ ]:
best_friendly_params = best_friendly_tree.get_params()
prunedFreindlyModel = DecisionTreeClassifier( random_state=42,min_samples_leaf = best_friendly_params["min_samples_leaf"], min_samples_split = best_friendly_params["min_samples_split"], ccp_alpha=1 / 10**1.9, criterion=best_friendly_params["criterion"])

start_time = time()
prunedFriendlyModel = prunedFreindlyModel.fit(Xtrain1, Ytrain1)
end_time = time()
prunedFriendlyTreeTime = end_time - start_time
# Visualizza l'albero decisionale
fig, ax = plt.subplots(figsize=(150, 100))
plot_tree(prunedFriendlyModel, filled=True, ax=ax)
plt.plot()

In [ ]:
# predizione del nuovo modello
Ypred1 = prunedFriendlyModel.predict(Xtest1)

cm = confusion_matrix(Ytest1, Ypred1)

print("Confusion matrix of the pruned model:\n", cm)
print("\nAccuracy of the pruned model:", cm.diagonal().sum() / cm.sum())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.show()

In [ ]:
# Valuta il modello utilizzando i dati di test

accuracy_train_test = accuracy_score(Ytest1, Ypred1)
precision = precision_score(Ytest1, Ypred1, pos_label=0)
recall = recall_score(Ytest1, Ypred1, pos_label=0)
f1 = f1_score(Ytest1, Ypred1, pos_label=0)

# Stampa le prestazioni del modello
print('Accuracy:', accuracy_train_test)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)

In [ ]:
print(classification_report(Ypred1, Ytest1))

In [ ]:
# Probabilità per la classe negativa
friendly_treeY_pred_prob_neg1 = prunedFriendlyModel.predict_proba(Xtest1)[:, 0]

# Calcola la curva ROC per i negativi
friendlyTreeFnr, friendlyTreeTnr, friendlyTreethresholds = roc_curve(Ytest1, friendly_treeY_pred_prob_neg1, pos_label=0)

# Calcola l'AUC della curva ROC
friendly_tree_roc_auc = roc_auc_score(Ytest1, friendly_treeY_pred_prob_neg1)

# Disegna la curva ROC
plt.plot(friendlyTreeFnr, friendlyTreeTnr, label='tree ROC curve (area = %0.2f)' % friendly_tree_roc_auc)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.ylabel('False Positive Rate')
plt.xlabel('True Positive Rate')
plt.title('Receiver operating characteristic (Negative Class)')
plt.legend(loc="lower right")
plt.show()


In [ ]:


# Funzione per costruire il modello Keras
def create_model(optimizer='adam', hidden_layer_sizes=(50,)):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))  # Definisci la forma di input
    # Aggiungi layer nascosti
    for units in hidden_layer_sizes:
        model.add(Dense(units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer per classificazione binaria
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Wrapping del modello Keras per scikit-learn
class KerasModel(BaseEstimator, ClassifierMixin):
    def __init__(self, optimizer='adam', hidden_layer_sizes=(50,)):
        self.optimizer = optimizer
        self.hidden_layer_sizes = hidden_layer_sizes
        self.model = create_model(optimizer=self.optimizer, hidden_layer_sizes=self.hidden_layer_sizes)

    def fit(self, X, y):
        self.model.fit(X, y, epochs=5, batch_size=32, verbose=0)
        return self

    def predict(self, X):
        return (self.model.predict(X) > 0.5).astype("int32")

    def predictProb(self, X):
        return self.model.predict(X)

# Caricamento dei dati (Xtrain e Ytrain devono essere già definiti)
X_train, X_val, y_train, y_val = train_test_split(Xtrain1, Ytrain1, test_size=0.2, random_state=42)

# Crea il modello Keras come un oggetto scikit-learn
keras_model = KerasModel()

# Definisci la griglia di parametri per il grid search
param_grid = {
    'optimizer': ['adam', 'sgd'],
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],  # Configurazioni dei layer nascosti
}

# Esegui il grid search con validazione incrociata (5-fold)
grid_search = GridSearchCV(estimator=keras_model, param_grid=param_grid, cv=3)

# Fitta il grid search sui dati di allenamento
grid_search.fit(X_train, y_train)

# Stampa i migliori iperparametri
print(f"Migliori iperparametri: {grid_search.best_params_}")

# Usa il modello con i migliori iperparametri per fare predizioni
best_friendly_net = grid_search.best_estimator_

# Fai le predizioni sul validation set
y_pred = best_friendly_net.predict(X_val)

# Calcola e stampa l'accuratezza sul validation set
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy sul validation set: {accuracy:.3f}")



In [ ]:
best_friendly_net.model.summary()

**Training di un modello dummy per valutare le preformance di training**

In [ ]:
YpredProb = best_friendly_net.predictProb(Xtest1)
Ypred13 = (YpredProb >= 0.5).astype(int)


accuracy_train_test = accuracy_score(Ytest1, Ypred13)
precision = precision_score(Ytest1, Ypred13, pos_label=0)
recall = recall_score(Ytest1, Ypred13, pos_label=0)
f1 = f1_score(Ytest1, Ypred13, pos_label=0)

# Stampa le prestazioni del modello
print('Accuracy:', accuracy_train_test)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)

In [ ]:
print(classification_report(Ypred13, Ytest1))

la precisione rimane comparabile a quando è stata effettuata riduzione di dimensionalità

In [ ]:
#perchè vogliamo lavorare sui negativi
YpredProbNegativa1 = 1 - YpredProb


# Calcola la curva ROC per la classe negativa (pos_label=0)
fnr, tnr, thresholds = roc_curve(Ytest1, YpredProbNegativa1, pos_label=0)


# Calcola l'AUC della curva ROC per la classe negativa
roc_auc = roc_auc_score(Ytest1, YpredProbNegativa1)

# Disegna la curva ROC per la classe negativa
plt.plot(friendlyTreeFnr, friendlyTreeTnr, label='Tree ROC curve (area = %0.2f)' % friendly_tree_roc_auc)
plt.plot(fnr, tnr, label='MLP ROC curve (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')  # Linea diagonale
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Negative Rate')
plt.ylabel('True Negative Rate')  # Tasso di veri negativi
plt.title('Receiver Operating Characteristic (Negative Class)')
plt.legend(loc="lower right")
plt.show()


**Analisi prestazionali in termini di risorse computazionali necessarie a trainare i due modelli, nelle differenti configurazione degli input**


Training dei due modelli di reti neurali per valutare le performance in termini di tempo di training.

In [ ]:
start_time = time()
# Trainiamo il modello
history = best_net.model.fit(Xtrain, Ytrain, epochs=5, batch_size=5, verbose=1)
end_time = time()
discriminatoryNetTime = end_time - start_time
# Valutiamo le prestazioni
score = best_net.model.evaluate(Xtest, Ytest, verbose=0)



In [ ]:
start_time = time()
# Trainiamo il modello
history = best_friendly_net.model.fit(Xtrain1, Ytrain1, epochs=5, batch_size=5, verbose=1)
end_time = time()
bestFriendlyNetTime = end_time - start_time
# Valutiamo le prestazioni
score = best_friendly_net.model.evaluate(Xtest1, Ytest1, verbose=0)

In [ ]:
print("Tempo di training normale: " + str(round(discriminatoryTreeTime, 2)) +  " secondi")
print("Tempo di training friendly: " + str(round(prunedFriendlyTreeTime, 2)) +  " secondi")
print("Tempo di training rete normale: " + str(round(discriminatoryNetTime, 2)) + " secondi")
print("Tempo di training rete friendly: " + str(round(bestFriendlyNetTime, 2)) + " secondi")

**L'albero è MOLTO più veloce della rete multistrato e l'accuracy di entrambi i modelli è nel margine di errore.**